In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
# In-silico digestion -> observable-peptide count (iBAQ denominator).
def read_fasta(path, keep):
    """Yield {header_id: sequence} for entries whose id is in `keep`."""
    seqs, pid, buf = {}, None, []
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                if pid in keep:
                    seqs[pid] = "".join(buf)
                pid = line[1:].strip().split()[0]
                buf = []
            else:
                buf.append(line.strip())
    if pid in keep:
        seqs[pid] = "".join(buf)
    return seqs


def digest_strict(seq):
    """Strict tryptic digest: cut C-terminal to every K/R (including before proline)."""
    cuts = [i + 1 for i, aa in enumerate(seq) if aa in "KR"]
    bounds = [0, *cuts, len(seq)]
    return [seq[bounds[i]:bounds[i + 1]] for i in range(len(bounds) - 1)]


def n_observable(seq, lo=6, hi=30):
    peps = [p for p in digest_strict(seq) if lo <= len(p) <= hi]
    return len(set(peps))




In [6]:
FASTA_PATH = Path("/cmnfs/ENB/search_results/FinalFragger/Okra/FASTA/2026-03-05-decoys-contam-GCA_035048815.1_Abelmoschus_esculentus.helixer.faa.fas")

In [ ]:
RESULTS_DIR = Path(
    "/cmnfs/ENB/search_results/FinalFragger/Okra/FragPipeSearchResults"
)

In [7]:
combined = pd.read_csv(
    RESULTS_DIR / "combined_protein.tsv", sep="\t", low_memory=False,
    usecols=["Protein"],
)
detected = set(combined.loc[~combined["Protein"].str.startswith("rev_"), "Protein"])

In [8]:
seqs = read_fasta(FASTA_PATH, detected)
print(f"Sequences matched: {len(seqs)} / {len(detected)}")

# Observable peptide count per protein; 0 -> NaN so iBAQ never divides by zero.
pep_counts = pd.Series({pid: n_observable(s) for pid, s in seqs.items()}, name="n_observable")
pep_counts_nz = pep_counts.replace(0, np.nan)
print(f"Proteins with 0 observable peptides: {int((pep_counts == 0).sum())}")
pep_counts.describe()

Sequences matched: 37165 / 37165
Proteins with 0 observable peptides: 1


count    37165.000000
mean        27.434549
std         19.528527
min          0.000000
25%         15.000000
50%         23.000000
75%         34.000000
max        401.000000
Name: n_observable, dtype: float64

In [ ]:
pep_counts[pep_counts == 0]

JAUCBN010000011.1_000563.1    0
Name: n_observable, dtype: int64